# Gemma-3-27B / Aya-Expanse-32B — APE preservation (headline + summary) (STANDALONE)

Zero-shot affect **preservation**: classify the model-generated **headline** and **summary**
(prompt noun changes accordingly), scored vs the gold article label. For
`google/gemma-3-27b-it` or `CohereForAI/aya-expanse-32b`. Pick the model, run top-to-bottom.

**Needs only** (in this folder): `ape_headline_bhojpuri.csv`, `ape_headline_maithili.csv`,
`ape_summary_bhojpuri.csv`, `ape_summary_maithili.csv`. 4-bit fits 24 GB. Outputs one bundle
covering all 4 slices (headline/summary × bho/mai) to send back.

## 1. Install

In [1]:
# %pip install -q "transformers>=4.51" "accelerate>=0.33" bitsandbytes torch pandas numpy scikit-learn tqdm sentencepiece

## 2. HF login (only for gated Gemma-3)

In [2]:
from huggingface_hub import login
login()  # uncomment for google/gemma-3-27b-it

## 3. Config

In [3]:
from pathlib import Path
import re

# ===== PICK ONE MODEL (comment/uncomment) =====
MODEL_ID = "google/gemma-3-27b-it"          # Google family (GATED: accept license + HF token)
# # MODEL_ID = "CohereForAI/aya-expanse-32b"  # Cohere family (ungated; multilingual)

LOAD_IN_4BIT   = True     # fits 24 GB (~16-19 GB). False = bf16 (needs 64-80 GB)
MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.1
MAX_TEXT_CHARS = 4000

_SAFE = re.sub(r"[^A-Za-z0-9._-]", "_", MODEL_ID)
OUT = Path(f"local_results_{_SAFE}")
CK  = Path(f"local_ckpt_{_SAFE}")
OUT.mkdir(exist_ok=True); CK.mkdir(exist_ok=True)
print("model:", MODEL_ID, "| 4bit:", LOAD_IN_4BIT)

model: google/gemma-3-27b-it | 4bit: True


## 4. Prompts + parsers

In [4]:
import json, re, math
import numpy as np

EMOTION_KEYS = ["ekman_joy","ekman_sadness","ekman_anger","ekman_fear",
                "ekman_disgust","ekman_surprise","ekman_neutral"]
EMOTIONS   = [k.replace("ekman_","") for k in EMOTION_KEYS]
SENTIMENTS = ["positive","negative","neutral"]
GOLD_EMO_COL, GOLD_SENT_COL = "dominant_emotion", "sentiment"

# USER prompts take a {noun}: "article" (source) | "headline" | "summary" (APE).
EMOTION_SYSTEM = (
    "You are an expert in affective computing and emotion analysis for "
    "multilingual Indian news content (Bhojpuri, Maithili, Hindi).\n\n"
    "Task: Analyze the emotional tone of the given news article using "
    "Ekman's 7 basic emotion categories.\n\n"
    "Rules:\n"
    "- All scores must be floats between 0.0 and 1.0\n"
    "- All 7 scores must sum to exactly 1.0\n"
    "- dominant_emotion must match the category with the highest score\n"
    "- Respond with ONLY valid JSON, no markdown, no explanation\n\n"
    'Required format:\n'
    '{\n'
    '  "ekman_joy":      0.00,\n'
    '  "ekman_sadness":  0.00,\n'
    '  "ekman_anger":    0.00,\n'
    '  "ekman_fear":     0.00,\n'
    '  "ekman_disgust":  0.00,\n'
    '  "ekman_surprise": 0.00,\n'
    '  "ekman_neutral":  0.00,\n'
    '  "dominant_emotion": "neutral"\n'
    '}'
)
EMOTION_USER = "Analyze the emotional tone of this news {noun}:\n\n{text}"

SENTIMENT_SYSTEM = (
    "You are an expert sentiment analyst specializing in multilingual Indian "
    "news content (Bhojpuri, Maithili, Hindi).\n\n"
    "Task: Classify the overall sentiment of the given news article.\n\n"
    "Rules:\n"
    "- Choose exactly one label: Positive, Negative, or Neutral\n"
    "- Positive  : good news, achievement, celebration, hope, or progress\n"
    "- Negative  : bad news, crime, disaster, conflict, grief, or criticism\n"
    "- Neutral   : factual/informational with no clear positive or negative tone\n"
    "- Respond with ONLY valid JSON, no markdown, no explanation\n\n"
    'Required format:\n{\n  "sentiment": "Positive"\n}'
)
SENTIMENT_USER = "Classify the sentiment of this news {noun}:\n\n{text}"

def _extract_json(raw, hint=""):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", " ", s, flags=re.DOTALL|re.IGNORECASE)
    s = re.sub(r"```(?:json)?", " ", s)
    cands = re.findall(r"\{[^{}]*\}", s, re.DOTALL)
    if hint:
        for c in cands:
            if hint in c: return c
    if cands: return cands[-1]
    m = re.search(r"\{.*\}", s, re.DOTALL)
    return m.group(0) if m else s

def parse_emotion(raw):
    if not raw: return None
    try:
        p = json.loads(_extract_json(raw, hint="ekman_joy"))
        if not all(k in p for k in EMOTION_KEYS): return None
        tot = sum(float(p[k]) for k in EMOTION_KEYS)
        if tot <= 0:
            dom = str(p.get("dominant_emotion","")).strip().lower()
            if dom in EMOTIONS:
                oh = {k:0.0 for k in EMOTION_KEYS}; oh[f"ekman_{dom}"]=1.0; oh["dominant_emotion"]=dom
                return oh
            return None
        for k in EMOTION_KEYS: p[k]=round(float(p[k])/tot,4)
        p["dominant_emotion"]=max(EMOTION_KEYS,key=lambda k:p[k]).replace("ekman_","")
        return {k:p[k] for k in EMOTION_KEYS+["dominant_emotion"]}
    except (json.JSONDecodeError, ValueError, KeyError, TypeError):
        return None

def parse_sentiment(raw):
    if not raw: return None
    try:
        lab = str(json.loads(_extract_json(raw, hint="sentiment")).get("sentiment","")).strip().lower()
        return lab if lab in SENTIMENTS else None
    except (json.JSONDecodeError, ValueError, KeyError, TypeError):
        return None

def js_divergence(p, q):
    p = np.clip(p,1e-12,None); p = p/p.sum()
    q = np.clip(q,1e-12,None); q = q/q.sum()
    m = 0.5*(p+q); kl = lambda a,b: float(np.sum(a*np.log2(a/b)))
    return 0.5*kl(p,m)+0.5*kl(q,m)

def safe_name(s): return re.sub(r"[^A-Za-z0-9._-]","_",s)
print("prompts + parsers ready")

prompts + parsers ready


## 5. Load model (4-bit)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

# Gemma-3-27b-it is a vision-language checkpoint -> AutoModelForCausalLM may not map.
# Fall back to the image-text-to-text class (text-only generation still works).
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
except Exception as e:
    print("AutoModelForCausalLM failed ->", str(e)[:120], "\ntrying AutoModelForImageTextToText ...")
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs)
model.eval()
if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
print("loaded:", MODEL_ID, "on", next(model.parameters()).device)

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

loaded: google/gemma-3-27b-it on cuda:0


## 6. Generate helper

In [6]:
# Gemma chat template has NO system role -> merge system into the user turn.
MERGE_SYSTEM = "gemma" in MODEL_ID.lower()

def _messages(system, user):
    if MERGE_SYSTEM:
        return [{"role":"user","content": system + "\n\n" + user}]
    return [{"role":"system","content":system},{"role":"user","content":user}]

@torch.inference_mode()
def generate(system, user):
    prompt = tok.apply_chat_template(_messages(system,user), tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=TEMPERATURE>0, temperature=max(TEMPERATURE,1e-5), top_p=0.9,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("smoke:", generate(SENTIMENT_SYSTEM, SENTIMENT_USER.format(noun="article", text="गाँव में खुशी के लहर, रिकॉर्ड फसल भइल।")))

smoke: ```json
{
  "sentiment": "Positive"
}
```


## 7. Score helper

In [7]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, confusion_matrix, classification_report)

def score_slice(task, lang, df, preds):
    """Return a metrics-detail dict (same schema as the API pipeline)."""
    g_emo,p_emo,js_list,g_sent,p_sent = [],[],[],[],[]
    n_missing=0
    for i in range(len(df)):
        if str(i) not in preds: n_missing+=1; continue
        pr=preds[str(i)]
        ge=str(df.at[i,GOLD_EMO_COL]).strip().lower(); pe=pr["dominant_emotion"]
        gvec=np.array([float(df.at[i,k]) if k in df.columns else 0.0 for k in EMOTION_KEYS])
        pvec=np.array([pr[k] for k in EMOTION_KEYS])
        js=js_divergence(gvec,pvec) if gvec.sum()>0 else float("nan")
        gs=str(df.at[i,GOLD_SENT_COL]).strip().lower() if GOLD_SENT_COL in df.columns else ""
        ps=pr.get("pred_sentiment","")
        g_emo.append(ge); p_emo.append(pe)
        if not math.isnan(js): js_list.append(js)
        if gs in SENTIMENTS: g_sent.append(gs); p_sent.append(ps)
    if not g_emo: return None
    el=sorted(set(g_emo)|set(p_emo))
    row={"task":task,"language":lang,"provider":"local","model":MODEL_ID,
         "n_scored":len(g_emo),"n_missing":n_missing,
         "emo_acc":accuracy_score(g_emo,p_emo),
         "emo_macro_precision":precision_score(g_emo,p_emo,average="macro",labels=el,zero_division=0),
         "emo_macro_recall":recall_score(g_emo,p_emo,average="macro",labels=el,zero_division=0),
         "emo_macro_f1":f1_score(g_emo,p_emo,average="macro",labels=el,zero_division=0),
         "emo_weighted_f1":f1_score(g_emo,p_emo,average="weighted",labels=el,zero_division=0),
         "emo_kappa":cohen_kappa_score(g_emo,p_emo),
         "emo_mean_js":float(np.mean(js_list)) if js_list else None}
    if g_sent:
        row.update({"sent_acc":accuracy_score(g_sent,p_sent),
            "sent_macro_precision":precision_score(g_sent,p_sent,average="macro",labels=SENTIMENTS,zero_division=0),
            "sent_macro_recall":recall_score(g_sent,p_sent,average="macro",labels=SENTIMENTS,zero_division=0),
            "sent_macro_f1":f1_score(g_sent,p_sent,average="macro",labels=SENTIMENTS,zero_division=0),
            "sent_weighted_f1":f1_score(g_sent,p_sent,average="weighted",labels=SENTIMENTS,zero_division=0),
            "sent_kappa":cohen_kappa_score(g_sent,p_sent)})
    row["emotion_confusion_matrix"]={"labels":el,"matrix":confusion_matrix(g_emo,p_emo,labels=el).tolist()}
    row["emotion_per_class"]=classification_report(g_emo,p_emo,labels=el,zero_division=0,output_dict=True)
    if g_sent:
        row["sentiment_confusion_matrix"]={"labels":SENTIMENTS,"matrix":confusion_matrix(g_sent,p_sent,labels=SENTIMENTS).tolist()}
        row["sentiment_per_class"]=classification_report(g_sent,p_sent,labels=SENTIMENTS,zero_division=0,output_dict=True)
    print(f"[{task}/{lang}] emo_acc={row['emo_acc']:.3f} sent_acc={row.get('sent_acc',float('nan')):.3f} (n={row['n_scored']}, missing={n_missing})")
    return row

## 8. Checkpointed classify

In [8]:
import pandas as pd, json
from pathlib import Path
from tqdm.auto import tqdm

def load_ckpt(p):  return json.load(open(p, encoding="utf-8")) if Path(p).exists() else {}
def save_ckpt(p, d):
    tmp=Path(p).with_suffix(".tmp"); json.dump(d, open(tmp,"w",encoding="utf-8"), ensure_ascii=False, indent=2); tmp.replace(p)

def classify(noun, text_col, input_csv, ckpt_path):
    df = pd.read_csv(input_csv); df.columns = df.columns.str.strip().str.lstrip("\ufeff")
    assert text_col in df.columns and GOLD_EMO_COL in df.columns, f"missing cols in {input_csv}"
    ck = load_ckpt(ckpt_path)
    pending=[i for i in range(len(df)) if str(i) not in ck]
    print(f"{input_csv}: total {len(df)} | done {len(ck)} | pending {len(pending)}")
    for n,i in enumerate(tqdm(pending, desc=Path(input_csv).stem)):
        text=str(df.at[i,text_col])[:MAX_TEXT_CHARS]
        emo=parse_emotion(generate(EMOTION_SYSTEM,  EMOTION_USER.format(noun=noun, text=text)))
        sent=parse_sentiment(generate(SENTIMENT_SYSTEM, SENTIMENT_USER.format(noun=noun, text=text)))
        if emo is None or sent is None: continue
        emo["pred_sentiment"]=sent; ck[str(i)]=emo
        if (n+1)%10==0: save_ckpt(ckpt_path, ck)
    save_ckpt(ckpt_path, ck)
    return df, ck

/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 9. Run APE tasks (headline + summary, both languages)

In [10]:
slices=[]
jobs = [
    ("headline","generated_headline","ape_headline_bhojpuri.csv","bhojpuri"),
    ("headline","generated_headline","ape_headline_maithili.csv","maithili"),
    ("summary","generated_summary","ape_summary_bhojpuri.csv","bhojpuri"),
    ("summary","generated_summary","ape_summary_maithili.csv","maithili"),
]
for task, col, csv, lang in jobs:
    ck_path = CK / f"{task}__{lang}.json"
    df, preds = classify(task, col, csv, ck_path)   # noun = task ("headline"/"summary")
    s = score_slice(task, lang, df, preds)
    if s: slices.append(s)


ape_headline_bhojpuri.csv: total 218 | done 0 | pending 218


ape_headline_bhojpuri:   0%|          | 0/218 [00:00<?, ?it/s]

[headline/bhojpuri] emo_acc=0.486 sent_acc=0.702 (n=218, missing=0)
ape_headline_maithili.csv: total 201 | done 0 | pending 201


ape_headline_maithili:   0%|          | 0/201 [00:00<?, ?it/s]

[headline/maithili] emo_acc=0.328 sent_acc=0.706 (n=201, missing=0)
ape_summary_bhojpuri.csv: total 218 | done 0 | pending 218


ape_summary_bhojpuri:   0%|          | 0/218 [00:00<?, ?it/s]

[summary/bhojpuri] emo_acc=0.546 sent_acc=0.761 (n=218, missing=0)
ape_summary_maithili.csv: total 201 | done 0 | pending 201


ape_summary_maithili:   0%|          | 0/201 [00:00<?, ?it/s]

[summary/maithili] emo_acc=0.383 sent_acc=0.711 (n=201, missing=0)


## 10. Bundle to send back

In [11]:
import json
bundle={"model":MODEL_ID,"provider":"local","task":"ape_preservation","slices":slices}
out = OUT / f"ape_bundle_{safe_name(MODEL_ID)}.json"
json.dump(bundle, open(out,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("SEND THIS BACK:", out)
print([(s["task"], s["language"], s["n_scored"]) for s in slices])

SEND THIS BACK: local_results_google_gemma-3-27b-it/ape_bundle_google_gemma-3-27b-it.json
[('headline', 'bhojpuri', 218), ('headline', 'maithili', 201), ('summary', 'bhojpuri', 218), ('summary', 'maithili', 201)]
